In [ ]:
# ============================================
# 02_limpieza.ipynb — Limpieza y normalización
# Proyecto: DataZ Movility (Fase A — Bizi)
# Autor: Miguel
# ============================================

import pandas as pd
import geopandas as gpd
from pathlib import Path

# Rutas del proyecto
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

print("Entorno cargado correctamente.")


In [ ]:
bizi_path = DATA_RAW / "bizi.geojson"

try:
    gdf = gpd.read_file(bizi_path)
    print("Dataset cargado correctamente.")
except Exception as e:
    print("Error al cargar el dataset:", e)

gdf.head()


In [ ]:
# Convertir lastUpdated a datetime
gdf["lastUpdated"] = pd.to_datetime(gdf["lastUpdated"], errors="coerce")

# Asegurar tipos numéricos
gdf["bicisDisponibles"] = pd.to_numeric(gdf["bicisDisponibles"], errors="coerce")
gdf["anclajesDisponibles"] = pd.to_numeric(gdf["anclajesDisponibles"], errors="coerce")

# Convertir estado a categoría
gdf["estado"] = gdf["estado"].astype("category")

gdf.dtypes


In [ ]:
gdf = gdf.rename(columns={
    "id": "station_id",
    "title": "station_name",
    "bicisDisponibles": "bikes",
    "anclajesDisponibles": "slots",
    "estado": "status"
})

gdf[["station_id", "station_name", "bikes", "slots", "status"]].head()


In [ ]:
# Asegurar CRS correcto
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)

# Extraer coordenadas
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

gdf[["station_id", "lon", "lat"]].head()


In [ ]:
import re

def clean_html(text):
    if pd.isna(text):
        return text
    return re.sub("<.*?>", "", text)

gdf["description_clean"] = gdf["description"].apply(clean_html)

gdf[["description", "description_clean"]].head()


In [ ]:
# Capacidad total
gdf["capacity"] = gdf["bikes"] + gdf["slots"]

# Ratio de ocupación
gdf["ratio_ocupacion"] = gdf["bikes"] / gdf["capacity"]

# Categoría de ocupación
def categorize_ratio(r):
    if pd.isna(r):
        return "desconocido"
    if r >= 0.7:
        return "alta"
    if r >= 0.3:
        return "media"
    return "baja"

gdf["ocupacion_categoria"] = gdf["ratio_ocupacion"].apply(categorize_ratio)

gdf[[
    "station_id", "bikes", "slots", "capacity",
    "ratio_ocupacion", "ocupacion_categoria"
]].head()


In [ ]:
anomalos = gdf[
    (gdf["bikes"] < 0) |
    (gdf["slots"] < 0) |
    (gdf["capacity"] <= 0)
]

anomalos


In [ ]:
cols_finales = [
    "station_id",
    "station_name",
    "status",
    "bikes",
    "slots",
    "capacity",
    "ratio_ocupacion",
    "ocupacion_categoria",
    "lon",
    "lat",
    "lastUpdated"
]

df_clean = gdf[cols_finales].copy()
df_clean.head()


In [ ]:
output_path = DATA_PROCESSED / "bizi_clean.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8")

print("Dataset limpio guardado en:", output_path)
